In [0]:
-- Create orders_clean only if it does not already exist.
-- WHERE 1 = 0 creates the table structure without loading any records.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`02-clean`.orders_clean AS
SELECT
    order_id,
    user_id,
    eval_set,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order -- Retain NULL values because they correspond to first orders.
FROM `ftw-week-06`.`01-raw`.orders
WHERE 1 = 0;


-- Incrementally load new or updated orders from the raw table.
-- MERGE prevents duplicate order_id values and updates existing records
-- if their data has changed.
MERGE INTO `ftw-week-06`.`02-clean`.orders_clean AS target
USING `ftw-week-06`.`01-raw`.orders AS source
ON target.order_id = source.order_id

-- Update the existing order when the same order_id is found in RAW.
WHEN MATCHED THEN UPDATE SET
    target.user_id = source.user_id,
    target.eval_set = source.eval_set,
    target.order_number = source.order_number,
    target.order_dow = source.order_dow,
    target.order_hour_of_day = source.order_hour_of_day,
    target.days_since_prior_order = source.days_since_prior_order

-- Insert the order when its order_id does not yet exist in CLEAN.
WHEN NOT MATCHED THEN INSERT (
    order_id,
    user_id,
    eval_set,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order
)
VALUES (
    source.order_id,
    source.user_id,
    source.eval_set,
    source.order_number,
    source.order_dow,
    source.order_hour_of_day,
    source.days_since_prior_order
);

In [0]:
-- addressed github comment by sara: retain the nulls in the clean layer (row 6816)
-- Create products_clean only if it does not already exist.
-- WHERE 1 = 0 creates the table structure without loading any records.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`02-clean`.products_clean AS
SELECT
    product_id,
    product_name,
    aisle_id,
    department_id
FROM `ftw-week-06`.`01-raw`.products
WHERE 1 = 0;


-- Incrementally load new or updated products from the raw table.
-- MERGE prevents duplicate product_id values and updates existing
-- products if their information has changed.
MERGE INTO `ftw-week-06`.`02-clean`.products_clean AS target
USING `ftw-week-06`.`01-raw`.products AS source
ON target.product_id = source.product_id

-- Update the existing product when the same product_id is found in RAW.
WHEN MATCHED THEN UPDATE SET
    target.product_name = source.product_name,
    target.aisle_id = source.aisle_id,
    target.department_id = source.department_id

-- Insert the product when its product_id does not yet exist in CLEAN.
WHEN NOT MATCHED THEN INSERT (
    product_id,
    product_name,
    aisle_id,
    department_id
)
VALUES (
    source.product_id,
    source.product_name,
    source.aisle_id,
    source.department_id
);

In [0]:
-- Create departments_clean only if it does not already exist.
-- WHERE 1 = 0 creates the table structure without loading any records.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`02-clean`.departments_clean AS
SELECT
    department_id,
    TRIM(department) AS department
FROM `ftw-week-06`.`01-raw`.departments
WHERE 1 = 0;


-- Incrementally load new or updated departments from the raw table.
-- MERGE prevents duplicate department_id values and updates existing
-- departments if their name changes.
MERGE INTO `ftw-week-06`.`02-clean`.departments_clean AS target
USING `ftw-week-06`.`01-raw`.departments AS source
ON target.department_id = source.department_id

-- Update the existing department when the same department_id is found in RAW.
WHEN MATCHED THEN UPDATE SET
    target.department = TRIM(source.department)

-- Insert the department when its department_id does not yet exist in CLEAN.
WHEN NOT MATCHED THEN INSERT (
    department_id,
    department
)
VALUES (
    source.department_id,
    TRIM(source.department)
);


In [0]:
-- Create aisles_clean only if it does not already exist.
-- WHERE 1 = 0 creates the table structure without loading any records.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`02-clean`.aisles_clean AS
SELECT
    aisle_id,
    TRIM(aisle) AS aisle
FROM `ftw-week-06`.`01-raw`.aisles
WHERE 1 = 0;


-- Incrementally load new or updated aisles from the raw table.
-- MERGE prevents duplicate aisle_id values and updates existing records
-- if the aisle name has changed.
MERGE INTO `ftw-week-06`.`02-clean`.aisles_clean AS target
USING `ftw-week-06`.`01-raw`.aisles AS source
ON target.aisle_id = source.aisle_id

-- Update the aisle name if the aisle_id already exists.
WHEN MATCHED THEN UPDATE SET
    target.aisle = TRIM(source.aisle)

-- Insert the aisle if the aisle_id does not yet exist.
WHEN NOT MATCHED THEN INSERT (
    aisle_id,
    aisle
)
VALUES (
    source.aisle_id,
    TRIM(source.aisle)
);

In [0]:
-- Create the table structure if it does not exist.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`02-clean`.order_products_clean AS
SELECT
    order_id,
    product_id,
    add_to_cart_order,
    reordered
FROM `ftw-week-06`.`01-raw`.order_products_prior
WHERE 1 = 0;


-- Insert only order-product records that do not already exist.
-- order_id + product_id serve as the composite key.
INSERT INTO `ftw-week-06`.`02-clean`.order_products_clean (
    order_id,
    product_id,
    add_to_cart_order,
    reordered
)
SELECT
    s.order_id,
    s.product_id,
    s.add_to_cart_order,
    s.reordered
FROM (
    SELECT
        order_id,
        product_id,
        add_to_cart_order,
        reordered
    FROM `ftw-week-06`.`01-raw`.order_products_prior

    UNION

    SELECT
        order_id,
        product_id,
        add_to_cart_order,
        reordered
    FROM `ftw-week-06`.`01-raw`.order_products_train
) s
WHERE NOT EXISTS (
    SELECT t.order_id
    FROM `ftw-week-06`.`02-clean`.order_products_clean t
    WHERE t.order_id = s.order_id
      AND t.product_id = s.product_id
);